# Obtención de los datos
Nos basaremos principalmente en dos fuentes de datos:
- El dataset ya existente de partidos internacionales en el repositorio:
- El elo proveniente de EloRatings.net

In [1]:
import pandas as pd
import requests
from pathlib import Path

# Constantes de URLs y rutas de archivos
BASE_URL = "https://www.eloratings.net"
HEADERS = {"User-Agent": "Mozilla/5.0"}
RESULTS_URL = "https://raw.githubusercontent.com/martj42/international_results/refs/heads/master/results.csv"
RAW_RESULTS_PATH = Path("../data/raw/results.csv")
PROCESSED_ELO_PATH = Path("../data/processed/historical_matches_elo.csv")


In [2]:

def descargar_tsv(nombre_archivo):
    response = requests.get(
        f"{BASE_URL}/{nombre_archivo}",
        timeout=30,
        headers=HEADERS,
    )
    response.raise_for_status()
    return response.text.splitlines()


def cargar_nombres_equipos():
    nombres_por_codigo = {}

    for linea in descargar_tsv("en.teams.tsv"):
        if not linea.strip():
            continue

        partes = [parte.strip() for parte in linea.split("\t")]
        codigo = partes[0]
        nombres = [nombre for nombre in partes[1:] if nombre]

        if codigo and nombres:
            nombres_por_codigo[codigo] = nombres[0]

    return nombres_por_codigo


def descargar_results_csv():
    print("Descargando historial de partidos...")

    response = requests.get(
        RESULTS_URL,
        timeout=30,
        headers=HEADERS,
    )
    response.raise_for_status()

    RAW_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    RAW_RESULTS_PATH.write_text(response.text, encoding="utf-8")

    return pd.read_csv(RAW_RESULTS_PATH)

def obtener_elo_actual():
    print("Descargando datos de EloRatings.net...")

    equipos_data = []
    nombres_por_codigo = cargar_nombres_equipos()

    for linea in descargar_tsv("World.tsv"):
        try:
            campos = [campo.strip() for campo in linea.split("\t")]
            if len(campos) < 4:
                continue

            codigo_equipo = campos[2]
            elo_texto = campos[3].replace(",", "").replace("−", "-")

            if codigo_equipo not in nombres_por_codigo or not elo_texto.lstrip("-").isdigit():
                continue

            pais = nombres_por_codigo[codigo_equipo]
            elo = int(elo_texto)

            if pais and elo:
                equipos_data.append({"Equipo": pais, "ELO_Oficial": elo})
        except (TypeError, ValueError):
            continue

    df_elo_oficial = pd.DataFrame(equipos_data)

    if df_elo_oficial.empty:
        raise ValueError("No se pudieron extraer filas de la tabla principal de EloRatings.")

    return df_elo_oficial


In [3]:
df_results = descargar_results_csv()
df_eloratings = obtener_elo_actual()

print("\n--- RESULTS CSV DESCARGADO ---")
print(df_results.head(5))
print("\n--- TOP 10 ELO RATINGS (DATOS EXTRAIDOS) ---")
print(df_eloratings.head(10))

PROCESSED_ELO_PATH.parent.mkdir(parents=True, exist_ok=True)
df_eloratings.to_csv(PROCESSED_ELO_PATH, index=False)

print(f"\nArchivo guardado en: {RAW_RESULTS_PATH}")
print(f"Archivo guardado en: {PROCESSED_ELO_PATH}")

Descargando historial de partidos...
Descargando datos de EloRatings.net...

--- RESULTS CSV DESCARGADO ---
         date home_team away_team  home_score  away_score tournament     city  \
0  1872-11-30  Scotland   England         0.0         0.0   Friendly  Glasgow   
1  1873-03-08   England  Scotland         4.0         2.0   Friendly   London   
2  1874-03-07  Scotland   England         2.0         1.0   Friendly  Glasgow   
3  1875-03-06   England  Scotland         2.0         2.0   Friendly   London   
4  1876-03-04  Scotland   England         3.0         0.0   Friendly  Glasgow   

    country  neutral  
0  Scotland    False  
1   England    False  
2  Scotland    False  
3   England    False  
4  Scotland    False  

--- TOP 10 ELO RATINGS (DATOS EXTRAIDOS) ---
        Equipo  ELO_Oficial
0        Spain         2165
1    Argentina         2113
2       France         2081
3      England         2020
4       Brazil         1984
5     Portugal         1984
6     Colombia         19